<a href="https://colab.research.google.com/github/SANGHATI23/ohdsi-fhir-omop-showcase-demo/blob/main/FHIRy_pyOMOP_TFL_PostPhaseE_WarningFix_v9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TFL Warning-Rule Correction

This notebook begins after the six fresh V0–V5 OMOP transformations.

It updates only the deterministic V2 and V3 warning logic and recomputes TFL metrics. Existing fresh OMOP databases are reused.

# Phase A

## Load saved run state

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import defaultdict
from enum import Enum
import re, ast
import numpy as np
import pandas as pd

MYDRIVE = Path("/content/drive/MyDrive")
REPO_DIR = Path("/content/ohdsi-fhir-omop-showcase-demo")

RUN_ROOT = MYDRIVE / "fhir_omop_colab" / "tfl_execution_v6"
NORMALIZED_DIR = RUN_ROOT / "normalized_sources"
AUDIT_DIR = RUN_ROOT / "fidelity_audit"

OUT_ROOT = REPO_DIR / "outputs"
FIDELITY_METRIC_OUT = OUT_ROOT / "fidelity_metrics"
FIDELITY_AUDIT_OUT = OUT_ROOT / "fidelity_audit"

for p in [FIDELITY_METRIC_OUT, FIDELITY_AUDIT_OUT]:
    p.mkdir(parents=True, exist_ok=True)

print("RUN_ROOT:", RUN_ROOT)

Mounted at /content/drive
RUN_ROOT: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6


In [2]:
def pick_normalized_file(variant):
    options = [
        NORMALIZED_DIR / f"{variant}_normalized.parquet",
        NORMALIZED_DIR / f"{variant}_recovered_normalized.parquet",
        NORMALIZED_DIR / f"{variant}_constructed_normalized.parquet",
    ]
    if variant == "V0":
        options.insert(0, NORMALIZED_DIR / "V0_recovered_normalized.parquet")
    if variant == "V5":
        options.insert(0, NORMALIZED_DIR / "V5_constructed_normalized.parquet")
    for p in options:
        if p.exists() and p.stat().st_size > 0:
            return p
    return None

SOURCE_DF = globals().get("SOURCE_DF", {})

for variant in ["V0","V1","V2","V3","V4","V5"]:
    if variant not in SOURCE_DF:
        path = pick_normalized_file(variant)
        if path is None:
            raise FileNotFoundError(
                f"Normalized source for {variant} was not found in {NORMALIZED_DIR}"
            )
        print("Loading", variant, "from", path.name)
        SOURCE_DF[variant] = pd.read_parquet(path)

print("\nNormalized sources ready:")
for variant in ["V0","V1","V2","V3","V4","V5"]:
    print(variant, f"{len(SOURCE_DF[variant]):,}")

Loading V0 from V0_recovered_normalized.parquet
Loading V1 from V1_normalized.parquet
Loading V2 from V2_normalized.parquet
Loading V3 from V3_normalized.parquet
Loading V4 from V4_normalized.parquet
Loading V5 from V5_constructed_normalized.parquet

Normalized sources ready:
V0 129,333
V1 129,333
V2 129,333
V3 129,333
V4 126,507
V5 129,333


In [3]:
TFL_AUDIT = globals().get("TFL_AUDIT", {})

for variant in ["V0","V1","V2","V3","V4","V5"]:
    if variant not in TFL_AUDIT:
        path = AUDIT_DIR / f"{variant}_fidelity_audit.parquet"
        if not path.exists() or path.stat().st_size == 0:
            raise FileNotFoundError(f"Existing TFL audit not found: {path}")
        print("Loading audit", variant)
        TFL_AUDIT[variant] = pd.read_parquet(path)

print("\nAudit rows:")
for variant in ["V0","V1","V2","V3","V4","V5"]:
    print(variant, f"{len(TFL_AUDIT[variant]):,}")

Loading audit V0
Loading audit V1
Loading audit V2
Loading audit V3
Loading audit V4
Loading audit V5

Audit rows:
V0 154,333
V1 154,333
V2 154,333
V3 154,333
V4 151,507
V5 154,333


# Phase B

## Resolve V2 Encounter identifier field

In [6]:
# ============================================================
# V2 raw Encounter recovery from original source archives
# ============================================================

from pathlib import Path
from collections import Counter
import tarfile
import json
import pandas as pd


# IMPORTANT: the folder name has a trailing space
RAW_ROOT = Path(
    "/content/drive/MyDrive/MyDrive fhir_omop_colab "
)


ARCHIVES = {
    "V1": RAW_ROOT / "V1_missing_demographics_clinical_core_25k.tar.gz",
    "V2": RAW_ROOT / "V2_duplicate_encounter_ids_clinical_core_25k.tar.gz",
    "V3": RAW_ROOT / "V3_conflicting_codings_clinical_core_25k.tar.gz",
    "V4": RAW_ROOT / "V4_missing_medications_clinical_core_25k.tar.gz",
}


# ------------------------------------------------------------
# Confirm archives
# ------------------------------------------------------------

print("RAW_ROOT exists:", RAW_ROOT.exists())
print()

for variant, archive in ARCHIVES.items():
    print(
        variant,
        "->",
        archive,
        "| exists:",
        archive.exists(),
        "| MB:",
        round(
            archive.stat().st_size / (1024 ** 2),
            2
        )
        if archive.exists()
        else None
    )


missing = [
    v
    for v, p in ARCHIVES.items()
    if not p.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing archive(s): "
        + ", ".join(missing)
    )


# ------------------------------------------------------------
# Show NDJSON members inside each archive
# ------------------------------------------------------------

archive_inventory = []


for variant, archive_path in ARCHIVES.items():

    with tarfile.open(
        archive_path,
        mode="r:gz"
    ) as tar:

        members = [
            m
            for m in tar.getmembers()
            if m.isfile()
        ]

        ndjson_members = [
            m.name
            for m in members
            if m.name.lower().endswith(
                ".ndjson"
            )
        ]

        encounter_named = [
            name
            for name in ndjson_members
            if "encounter" in name.lower()
        ]


        archive_inventory.append({
            "variant": variant,
            "files": len(members),
            "ndjson_files": len(
                ndjson_members
            ),
            "encounter_named_files": len(
                encounter_named
            ),
            "encounter_files": " | ".join(
                encounter_named
            ),
        })


archive_inventory = pd.DataFrame(
    archive_inventory
)

print(
    "\nArchive inventory:"
)

display(
    archive_inventory
)


# ------------------------------------------------------------
# Read Encounter.id directly from archive
#
# First prefer NDJSON whose filename contains Encounter.
# If filename is generic, examine all NDJSON resources.
# ------------------------------------------------------------

def read_encounter_ids_from_archive(
    archive_path
):

    ids = []

    with tarfile.open(
        archive_path,
        mode="r:gz"
    ) as tar:

        members = [
            m
            for m in tar.getmembers()
            if (
                m.isfile()
                and
                m.name.lower().endswith(
                    ".ndjson"
                )
            )
        ]


        encounter_members = [
            m
            for m in members
            if "encounter" in m.name.lower()
        ]


        # If no explicit Encounter filename exists,
        # inspect all NDJSON members.
        members_to_scan = (
            encounter_members
            if encounter_members
            else members
        )


        for member in members_to_scan:

            handle = tar.extractfile(
                member
            )

            if handle is None:
                continue


            for raw_line in handle:

                try:

                    line = raw_line.decode(
                        "utf-8"
                    ).strip()

                except Exception:

                    continue


                if not line:
                    continue


                try:

                    obj = json.loads(
                        line
                    )

                except Exception:

                    continue


                # Standard Bulk FHIR NDJSON
                if (
                    obj.get(
                        "resourceType"
                    )
                    ==
                    "Encounter"
                ):

                    encounter_id = obj.get(
                        "id"
                    )

                    if encounter_id is not None:

                        ids.append(
                            str(
                                encounter_id
                            )
                        )


                # Also tolerate Bundle input
                elif (
                    obj.get(
                        "resourceType"
                    )
                    ==
                    "Bundle"
                ):

                    for entry in obj.get(
                        "entry",
                        []
                    ):

                        resource = entry.get(
                            "resource",
                            {}
                        )

                        if (
                            resource.get(
                                "resourceType"
                            )
                            ==
                            "Encounter"
                        ):

                            encounter_id = (
                                resource.get(
                                    "id"
                                )
                            )

                            if (
                                encounter_id
                                is not None
                            ):

                                ids.append(
                                    str(
                                        encounter_id
                                    )
                                )


    return ids


# ------------------------------------------------------------
# Compare V1/V2/V3/V4
# ------------------------------------------------------------

RAW_ENCOUNTER_IDS = {}

summary_rows = []


for variant, archive_path in ARCHIVES.items():

    print(
        "\nReading",
        variant,
        "Encounter IDs..."
    )

    ids = (
        read_encounter_ids_from_archive(
            archive_path
        )
    )

    RAW_ENCOUNTER_IDS[
        variant
    ] = ids


    counts = Counter(
        ids
    )


    duplicate_counts = {
        encounter_id: n
        for encounter_id, n
        in counts.items()
        if n > 1
    }


    duplicate_rows = sum(
        duplicate_counts.values()
    )


    summary_rows.append({
        "variant":
            variant,

        "encounter_rows":
            len(ids),

        "unique_encounter_ids":
            len(counts),

        "duplicate_id_values":
            len(
                duplicate_counts
            ),

        "duplicate_rows":
            duplicate_rows,
    })


raw_encounter_summary = pd.DataFrame(
    summary_rows
)


print(
    "\nRAW FHIR ENCOUNTER SUMMARY"
)

display(
    raw_encounter_summary
)


# ------------------------------------------------------------
# V2 duplicate IDs
# ------------------------------------------------------------

v2_counts = Counter(
    RAW_ENCOUNTER_IDS[
        "V2"
    ]
)


V2_RAW_DUPLICATE_IDS = {
    encounter_id
    for encounter_id, n
    in v2_counts.items()
    if n > 1
}


# ------------------------------------------------------------
# Validate controlled perturbation
# ------------------------------------------------------------

baseline_rows = (
    raw_encounter_summary[
        raw_encounter_summary[
            "variant"
        ].isin(
            [
                "V1",
                "V3",
                "V4",
            ]
        )
    ][
        "duplicate_rows"
    ]
)


v2_duplicate_rows = int(
    raw_encounter_summary.loc[
        raw_encounter_summary[
            "variant"
        ]
        ==
        "V2",
        "duplicate_rows"
    ].iloc[0]
)


baseline_max = int(
    baseline_rows.max()
)


print(
    "\nBaseline max duplicate rows:",
    baseline_max
)

print(
    "V2 duplicate rows:",
    v2_duplicate_rows
)

print(
    "V2 duplicate ID values:",
    len(
        V2_RAW_DUPLICATE_IDS
    )
)


if v2_duplicate_rows <= baseline_max:

    raise RuntimeError(
        "V2 raw Encounter duplication is not greater "
        "than V1/V3/V4. Stop here."
    )


V2_RESOLUTION_MODE = (
    "raw_fhir_encounter_id"
)


print(
    "\nPASS: V2 controlled Encounter-ID "
    "perturbation verified in original FHIR."
)

print(
    "V2_RESOLUTION_MODE =",
    V2_RESOLUTION_MODE
)

RAW_ROOT exists: True

V1 -> /content/drive/MyDrive/MyDrive fhir_omop_colab /V1_missing_demographics_clinical_core_25k.tar.gz | exists: True | MB: 8.35
V2 -> /content/drive/MyDrive/MyDrive fhir_omop_colab /V2_duplicate_encounter_ids_clinical_core_25k.tar.gz | exists: True | MB: 8.35
V3 -> /content/drive/MyDrive/MyDrive fhir_omop_colab /V3_conflicting_codings_clinical_core_25k.tar.gz | exists: True | MB: 8.37
V4 -> /content/drive/MyDrive/MyDrive fhir_omop_colab /V4_missing_medications_clinical_core_25k.tar.gz | exists: True | MB: 8.27

Archive inventory:


,variant,files,ndjson_files,encounter_named_files,encounter_files
0,V1,8,8,1,V1_missing_demographics_clinical_core_25k/Enco...
1,V2,8,8,8,V2_duplicate_encounter_ids_clinical_core_25k/P...
2,V3,8,8,1,V3_conflicting_codings_clinical_core_25k/Encou...
3,V4,8,8,1,V4_missing_medications_clinical_core_25k/Encou...



Reading V1 Encounter IDs...

Reading V2 Encounter IDs...

Reading V3 Encounter IDs...

Reading V4 Encounter IDs...

RAW FHIR ENCOUNTER SUMMARY


,variant,encounter_rows,unique_encounter_ids,duplicate_id_values,duplicate_rows
0,V1,25000,25000,0,0
1,V2,25000,23790,1183,2393
2,V3,25000,25000,0,0
3,V4,25000,25000,0,0



Baseline max duplicate rows: 0
V2 duplicate rows: 2393
V2 duplicate ID values: 1183

PASS: V2 controlled Encounter-ID perturbation verified in original FHIR.
V2_RESOLUTION_MODE = raw_fhir_encounter_id


## Resolve V3 Condition coding fields

In [7]:
def parse_multi_value(value):
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    if isinstance(value, (list, tuple, set, np.ndarray)):
        values = list(value)
    else:
        text = str(value).strip()
        if text == "" or text.lower() in {"nan","none","null","<na>"}:
            return []
        values = None
        if (
            (text.startswith("[") and text.endswith("]"))
            or (text.startswith("(") and text.endswith(")"))
        ):
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, (list, tuple, set)):
                    values = list(parsed)
            except Exception:
                values = None
        if values is None:
            values = [text]

    return [
        str(x).strip().strip("'\"")
        for x in values
        if str(x).strip() not in {"","nan","None","<NA>"}
    ]

def direct_condition_coding_columns(df):
    code_columns, system_columns = [], []
    for c in df.columns:
        low = c.lower()
        if low.startswith("code.coding.") or low.startswith("resource.code.coding."):
            if low.endswith(".code") or low.endswith(".codes"):
                code_columns.append(c)
            if low.endswith(".system") or low.endswith(".systems"):
                system_columns.append(c)
    return sorted(code_columns), sorted(system_columns)

condition_v0 = resource_subset(SOURCE_DF["V0"], "Condition")
CONDITION_CODE_COLUMNS, CONDITION_SYSTEM_COLUMNS = direct_condition_coding_columns(condition_v0)

print("Condition code columns:")
for c in CONDITION_CODE_COLUMNS:
    print(" -", c)

print("\nCondition system columns:")
for c in CONDITION_SYSTEM_COLUMNS:
    print(" -", c)

if not CONDITION_CODE_COLUMNS:
    raise RuntimeError("No direct Condition.code.coding code column was found.")

Condition code columns:
 - code.coding.codes

Condition system columns:


In [9]:
# ============================================================
# Resolve V3 conflicting Condition coding representations
# ============================================================

import ast
import json
import numpy as np
import pandas as pd


def resource_subset(df, resource_type):
    return df[
        df["__tfl_resource_type"].astype(str)
        ==
        resource_type
    ].copy()


# ------------------------------------------------------------
# Convert a flattened coding value into comparable tokens
# ------------------------------------------------------------

def flatten_tokens(value):

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass


    # Already structured
    if isinstance(value, dict):

        tokens = []

        for k, v in value.items():

            if isinstance(
                v,
                (
                    dict,
                    list,
                    tuple,
                    set,
                    np.ndarray,
                )
            ):
                tokens.extend(
                    flatten_tokens(v)
                )

            elif v is not None:

                text = str(v).strip()

                if text:
                    tokens.append(text)

        return tokens


    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            np.ndarray,
        )
    ):

        tokens = []

        for item in value:
            tokens.extend(
                flatten_tokens(item)
            )

        return tokens


    text = str(value).strip()


    if (
        not text
        or
        text.lower()
        in {
            "nan",
            "none",
            "null",
            "<na>",
        }
    ):
        return []


    # Stringified Python/JSON list/dict
    if (
        (
            text.startswith("[")
            and text.endswith("]")
        )
        or
        (
            text.startswith("{")
            and text.endswith("}")
        )
        or
        (
            text.startswith("(")
            and text.endswith(")")
        )
    ):

        parsed = None

        try:
            parsed = ast.literal_eval(
                text
            )

        except Exception:

            try:
                parsed = json.loads(
                    text
                )

            except Exception:
                parsed = None


        if parsed is not None:

            return flatten_tokens(
                parsed
            )


    # Treat the normalized scalar representation as one code value.
    return [text]


def tokens_for_column(
    df,
    column
):

    tokens = []

    for value in df[column]:

        tokens.extend(
            flatten_tokens(value)
        )


    return {
        str(x).strip()
        for x in tokens
        if str(x).strip()
    }


# ============================================================
# STEP 1
# Find coding-related Condition columns.
# ============================================================

CONDITION = {
    variant: resource_subset(
        SOURCE_DF[variant],
        "Condition"
    )
    for variant in [
        "V0",
        "V1",
        "V2",
        "V3",
        "V4",
        "V5",
    ]
}


common_columns = sorted(
    set.intersection(
        *[
            set(df.columns)
            for df in CONDITION.values()
        ]
    )
)


coding_candidates = [
    c
    for c in common_columns

    if not c.startswith(
        "__tfl_"
    )

    and any(
        token in c.lower()
        for token in [
            "code",
            "coding",
            "system",
        ]
    )
]


print(
    "Condition coding-related columns:",
    len(
        coding_candidates
    )
)


# ============================================================
# STEP 2
# Determine which coding field reproduces the controlled
# V3/V5 pattern.
#
# Required pattern:
#
# V1 = V0
# V2 = V0
# V4 = V0
#
# V3 > V0
# V5 > V0
# ============================================================

diagnostic_rows = []

FIELD_TOKEN_SETS = {}


for column in coding_candidates:

    variant_sets = {}

    failed = False


    for variant in [
        "V0",
        "V1",
        "V2",
        "V3",
        "V4",
        "V5",
    ]:

        try:

            token_set = tokens_for_column(
                CONDITION[variant],
                column
            )

        except Exception:

            failed = True
            break


        variant_sets[
            variant
        ] = token_set


    if failed:
        continue


    v0 = variant_sets[
        "V0"
    ]


    if len(v0) == 0:
        continue


    unaffected_equal = (

        variant_sets["V1"]
        ==
        v0

        and

        variant_sets["V2"]
        ==
        v0

        and

        variant_sets["V4"]
        ==
        v0
    )


    v3_expanded = (
        len(
            variant_sets[
                "V3"
            ]
        )
        >
        len(v0)
    )


    v5_expanded = (
        len(
            variant_sets[
                "V5"
            ]
        )
        >
        len(v0)
    )


    v3_novel = (
        variant_sets["V3"]
        -
        v0
    )


    v5_novel = (
        variant_sets["V5"]
        -
        v0
    )


    # Prefer actual code paths over system/display paths.
    low = column.lower()

    preference = 0

    if "coding" in low:
        preference += 50

    if (
        low.endswith(".codes")
        or
        low.endswith(".code")
    ):
        preference += 50

    if "system" in low:
        preference += 10

    if "display" in low:
        preference -= 20


    diagnostic_rows.append({

        "column":
            column,

        "v0_unique":
            len(
                variant_sets[
                    "V0"
                ]
            ),

        "v1_unique":
            len(
                variant_sets[
                    "V1"
                ]
            ),

        "v2_unique":
            len(
                variant_sets[
                    "V2"
                ]
            ),

        "v3_unique":
            len(
                variant_sets[
                    "V3"
                ]
            ),

        "v4_unique":
            len(
                variant_sets[
                    "V4"
                ]
            ),

        "v5_unique":
            len(
                variant_sets[
                    "V5"
                ]
            ),

        "v3_delta":
            (
                len(
                    variant_sets[
                        "V3"
                    ]
                )
                -
                len(v0)
            ),

        "v5_delta":
            (
                len(
                    variant_sets[
                        "V5"
                    ]
                )
                -
                len(v0)
            ),

        "v3_novel_values":
            len(
                v3_novel
            ),

        "v5_novel_values":
            len(
                v5_novel
            ),

        "unaffected_equal_v0":
            unaffected_equal,

        "v3_expanded":
            v3_expanded,

        "v5_expanded":
            v5_expanded,

        "preference":
            preference,
    })


    FIELD_TOKEN_SETS[
        column
    ] = variant_sets


condition_coding_diagnostics = pd.DataFrame(
    diagnostic_rows
)


condition_coding_diagnostics = (
    condition_coding_diagnostics
    .sort_values(
        [
            "unaffected_equal_v0",
            "v3_expanded",
            "v5_expanded",
            "preference",
            "v3_delta",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            False,
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nCondition coding diagnostics:"
)

display(
    condition_coding_diagnostics.head(
        30
    )
)


# ============================================================
# STEP 3
# Select only a field that matches the controlled experiment.
# ============================================================

valid_fields = (
    condition_coding_diagnostics[
        condition_coding_diagnostics[
            "unaffected_equal_v0"
        ]
        &
        condition_coding_diagnostics[
            "v3_expanded"
        ]
        &
        condition_coding_diagnostics[
            "v5_expanded"
        ]
    ]
    .copy()
)


if valid_fields.empty:

    raise RuntimeError(
        "No normalized Condition coding field reproduces "
        "the required V3/V5 controlled pattern. "
        "Do not redefine the warning rule."
    )


# Highest preference, then largest V3 expansion.
valid_fields = valid_fields.sort_values(
    [
        "preference",
        "v3_delta",
        "v3_novel_values",
    ],
    ascending=[
        False,
        False,
        False,
    ]
)


V3_CONDITION_CODING_FIELD = str(
    valid_fields.iloc[0][
        "column"
    ]
)


print(
    "\nSelected Condition coding field:"
)

print(
    V3_CONDITION_CODING_FIELD
)


display(
    valid_fields.head(10)
)


# ============================================================
# STEP 4
# Define V0 coding representation set.
#
# The controlled warning is raised when a Condition contains
# a coding representation not present in the V0 baseline set.
# ============================================================

V0_CONDITION_CODE_SET = (
    FIELD_TOKEN_SETS[
        V3_CONDITION_CODING_FIELD
    ][
        "V0"
    ]
)


print(
    "\nV0 coding representations:",
    len(
        V0_CONDITION_CODE_SET
    )
)


# ============================================================
# STEP 5
# Record row-level competing representations.
# ============================================================

def novel_condition_coding_mask(
    df,
    coding_field,
    baseline_codes
):

    conditions = resource_subset(
        df,
        "Condition"
    )


    flags = []


    for value in conditions[
        coding_field
    ]:

        tokens = set(
            flatten_tokens(
                value
            )
        )


        novel = (
            tokens
            -
            baseline_codes
        )


        flags.append(
            len(novel)
            >
            0
        )


    return pd.Series(
        flags,
        index=conditions.index,
        dtype=bool
    )


CONDITION_CONFLICT_MASK = {}

summary_rows = []


for variant in [
    "V0",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
]:

    mask = (
        novel_condition_coding_mask(
            SOURCE_DF[
                variant
            ],
            V3_CONDITION_CODING_FIELD,
            V0_CONDITION_CODE_SET
        )
    )


    CONDITION_CONFLICT_MASK[
        variant
    ] = mask


    summary_rows.append({

        "variant":
            variant,

        "condition_rows":
            len(
                mask
            ),

        "conflicting_coding_rows":
            int(
                mask.sum()
            ),

        "warning_rate":
            (
                float(
                    mask.mean()
                )
                if len(mask)
                else np.nan
            ),
    })


condition_conflict_summary = pd.DataFrame(
    summary_rows
)


print(
    "\nV3 CONDITION WARNING SUMMARY"
)

display(
    condition_conflict_summary
)


# ============================================================
# STEP 6
# Controlled-validation gates
# ============================================================

counts = (
    condition_conflict_summary
    .set_index(
        "variant"
    )[
        "conflicting_coding_rows"
    ]
    .to_dict()
)


# V0 and unrelated single perturbations must remain clean.
for variant in [
    "V0",
    "V1",
    "V2",
    "V4",
]:

    if counts[
        variant
    ] != 0:

        raise RuntimeError(
            f"{variant} has "
            f"{counts[variant]} unexpected "
            "V3 coding warnings."
        )


# V3 must produce the coding warning.
if counts[
    "V3"
] <= 0:

    raise RuntimeError(
        "V3 did not produce any competing "
        "Condition coding warnings."
    )


# V5 contains the same V3 coding perturbation.
if counts[
    "V5"
] <= 0:

    raise RuntimeError(
        "V5 did not retain the V3 coding warning."
    )


print(
    "\nPASS: V3 competing coding representations "
    "detected above V0."
)

print(
    "PASS: V5 retains the V3 coding signal."
)

print(
    "PASS: V1, V2, and V4 remain clean for "
    "the V3 coding warning."
)

Condition coding-related columns: 12

Condition coding diagnostics:


,column,v0_unique,v1_unique,v2_unique,v3_unique,v4_unique,v5_unique,v3_delta,v5_delta,v3_novel_values,v5_novel_values,unaffected_equal_v0,v3_expanded,v5_expanded,preference
0,code.coding.codes,247,247,247,412,247,412,165,165,170,170,True,True,True,100
1,clinicalStatus.coding.codes,2,2,2,2,2,2,0,0,0,0,True,False,False,100
2,verificationStatus.coding.codes,1,1,1,1,1,1,0,0,0,0,True,False,False,100
3,code.text,247,247,247,247,247,247,0,0,0,0,True,False,False,0



Selected Condition coding field:
code.coding.codes


,column,v0_unique,v1_unique,v2_unique,v3_unique,v4_unique,v5_unique,v3_delta,v5_delta,v3_novel_values,v5_novel_values,unaffected_equal_v0,v3_expanded,v5_expanded,preference
0,code.coding.codes,247,247,247,412,247,412,165,165,170,170,True,True,True,100



V0 coding representations: 247

V3 CONDITION WARNING SUMMARY


,variant,condition_rows,conflicting_coding_rows,warning_rate
0,V0,25000,0,0.0000
1,V1,25000,0,0.0000
2,V2,25000,0,0.0000
3,V3,25000,2510,0.1004
4,V4,25000,0,0.0000
5,V5,25000,2510,0.1004



PASS: V3 competing coding representations detected above V0.
PASS: V5 retains the V3 coding signal.
PASS: V1, V2, and V4 remain clean for the V3 coding warning.


# Phase C

## Update V2 and V3 warning labels

In [12]:
# ============================================================
# Restore pyOMOP mapping rules for V2/V3 warning-label phase
# ============================================================

import sys
import subprocess
import importlib
import json
from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# 1. Ensure the same pyOMOP version used in the experiment
# ------------------------------------------------------------

try:
    import pyomop

except ModuleNotFoundError:

    print("pyomop not found. Installing pyomop==6.4.0 ...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pyomop==6.4.0"
    ])

    importlib.invalidate_caches()

    import pyomop


print(
    "pyomop version:",
    getattr(
        pyomop,
        "__version__",
        "unknown"
    )
)

print(
    "pyomop location:",
    pyomop.__file__
)


# ------------------------------------------------------------
# 2. Locate mapping.default.json
# ------------------------------------------------------------

PYOMOP_DIR = Path(
    pyomop.__file__
).resolve().parent


MAPPING_PATH = (
    PYOMOP_DIR
    / "mapping.default.json"
)


# Fallback in case package layout differs
if not MAPPING_PATH.exists():

    candidates = list(
        PYOMOP_DIR.rglob(
            "mapping.default.json"
        )
    )

    if len(candidates) == 1:

        MAPPING_PATH = candidates[0]

    elif len(candidates) > 1:

        # Prefer file closest to package root
        candidates = sorted(
            candidates,
            key=lambda p: len(
                p.relative_to(
                    PYOMOP_DIR
                ).parts
            )
        )

        MAPPING_PATH = candidates[0]

    else:

        raise FileNotFoundError(
            "Could not locate mapping.default.json "
            f"inside {PYOMOP_DIR}"
        )


print(
    "\nMapping file:",
    MAPPING_PATH
)


# ------------------------------------------------------------
# 3. Read pyOMOP mapping
# ------------------------------------------------------------

with open(
    MAPPING_PATH,
    "r",
    encoding="utf-8"
) as f:

    PYOMOP_MAPPING = json.load(
        f
    )


# ------------------------------------------------------------
# 4. Extract FHIR resource type from table filters
# ------------------------------------------------------------

def infer_resource_type(
    table_map
):

    filters = (
        table_map.get(
            "filters",
            []
        )
        or []
    )


    for flt in filters:

        column = flt.get(
            "column"
        )


        if (
            column
            in {
                "resourceType",
                "resource.resourceType"
            }
            and
            "equals"
            in flt
        ):

            return str(
                flt["equals"]
            )


    return None


# ------------------------------------------------------------
# 5. Rebuild mapping_rules_df
# ------------------------------------------------------------

mapping_rows = []


for ordinal, table_map in enumerate(
    PYOMOP_MAPPING.get(
        "tables",
        []
    ),
    start=1
):

    resource_type = (
        infer_resource_type(
            table_map
        )
    )


    if not resource_type:
        continue


    target_table = (
        table_map.get(
            "name"
        )
    )


    if not target_table:
        continue


    columns = (
        table_map.get(
            "columns",
            {}
        )
        or {}
    )


    mapping_rule_id = (
        f"{resource_type}"
        f"_to_"
        f"{target_table}"
        f"_r{ordinal:02d}"
    )


    target_fields = sorted(
        str(x)
        for x in columns.keys()
    )


    source_paths = sorted({
        str(value)

        for value
        in columns.values()

        if (
            isinstance(
                value,
                str
            )
            and
            value.strip()
        )
    })


    mapping_rows.append({

        "rule_order":
            ordinal,

        "source_resource_type":
            resource_type,

        "target_omop_table":
            target_table,

        "mapping_rule_id":
            mapping_rule_id,

        "target_fields":
            "|".join(
                target_fields
            ),

        "source_paths":
            "|".join(
                source_paths
            ),
    })


mapping_rules_df = pd.DataFrame(
    mapping_rows
)


if mapping_rules_df.empty:

    raise RuntimeError(
        "pyOMOP mapping rules could not be reconstructed."
    )


# ------------------------------------------------------------
# 6. Show the rules
# ------------------------------------------------------------

print(
    "\nMapping rules reconstructed:",
    len(
        mapping_rules_df
    )
)


display(
    mapping_rules_df
)


# ------------------------------------------------------------
# 7. Recreate missing output CSV
# ------------------------------------------------------------

if "REPO_DIR" not in globals():

    REPO_DIR = Path(
        "/content/ohdsi-fhir-omop-showcase-demo"
    )


MAPPING_OUT = (
    REPO_DIR
    / "outputs"
    / "mapping_summary"
)


MAPPING_OUT.mkdir(
    parents=True,
    exist_ok=True
)


mapping_csv = (
    MAPPING_OUT
    / "tfl_mapping_rules.csv"
)


mapping_rules_df.to_csv(
    mapping_csv,
    index=False
)


print(
    "\nPASS: mapping_rules_df restored."
)

print(
    "Saved:",
    mapping_csv
)

pyomop not found. Installing pyomop==6.4.0 ...


pyomop version: 6.4.0
pyomop location: /usr/local/lib/python3.12/dist-packages/pyomop/__init__.py

Mapping file: /usr/local/lib/python3.12/dist-packages/pyomop/mapping.default.json

Mapping rules reconstructed: 10


,rule_order,source_resource_type,target_omop_table,mapping_rule_id,target_fields,source_paths
0,1,Patient,person,Patient_to_person_r01,birth_datetime|ethnicity_concept_id|ethnicity_...,birthDate|extension|gender|patientId
1,2,Encounter,visit_occurrence,Encounter_to_visit_occurrence_r02,admitted_from_source_value|discharged_to_conce...,class.code|hospitalization.dischargeDispositio...
2,3,Condition,condition_occurrence,Condition_to_condition_occurrence_r03,condition_concept_id|condition_end_date|condit...,abatementDateTime|clinicalStatus.coding.codes|...
3,4,Procedure,procedure_occurrence,Procedure_to_procedure_occurrence_r04,modifier_concept_id|modifier_source_value|pers...,code.coding.codes|patientId|performedPeriod.en...
4,5,Device,device_exposure,Device_to_device_exposure_r05,device_concept_id|device_exposure_end_date|dev...,distinctIdentifier|expirationDate|manufactureD...
5,6,Observation,measurement,Observation_to_measurement_r06,measurement_concept_id|measurement_date|measur...,code.coding.codes|effectiveDateTime|issued|pat...
6,7,Observation,observation,Observation_to_observation_r07,observation_concept_id|observation_date|observ...,code.coding.codes|effectiveDateTime|issued|pat...
7,8,AllergyIntolerance,observation,AllergyIntolerance_to_observation_r08,observation_concept_id|observation_date|observ...,code.coding.codes|patientId|reaction|recordedDate
8,9,Immunization,drug_exposure,Immunization_to_drug_exposure_r09,drug_concept_id|drug_exposure_end_date|drug_ex...,occurrenceDateTime|patientId|vaccineCode.codin...
9,10,MedicationRequest,drug_exposure,MedicationRequest_to_drug_exposure_r10,drug_concept_id|drug_exposure_end_date|drug_ex...,authoredOn|dosageInstruction|medicationCodeabl...



PASS: mapping_rules_df restored.
Saved: /content/ohdsi-fhir-omop-showcase-demo/outputs/mapping_summary/tfl_mapping_rules.csv


In [14]:
# ============================================================
# Phase C — Final V2 validation summary
# Works with either:
#   1. normalized Encounter duplicate detection
#   2. raw FHIR Encounter.id verification
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# CASE 1
# Duplicate Encounter rows were resolved in normalized data
# ------------------------------------------------------------

if (
    "V2_ENCOUNTER_DUPLICATE_SOURCE_INDEX"
    in globals()
):

    v2_duplicate_summary = pd.DataFrame([
        {
            "variant": variant,
            "encounter_duplicate_rows": len(
                V2_ENCOUNTER_DUPLICATE_SOURCE_INDEX.get(
                    variant,
                    set()
                )
            ),
            "resolution_source": "normalized_dataframe",
        }

        for variant in [
            "V0",
            "V1",
            "V2",
            "V3",
            "V4",
            "V5",
        ]
    ])


# ------------------------------------------------------------
# CASE 2
# V2 Encounter identity was verified from raw FHIR
# ------------------------------------------------------------

elif (
    "raw_encounter_summary"
    in globals()
):

    raw_summary = (
        raw_encounter_summary
        .copy()
    )


    # Standardize column name
    if "duplicate_rows" not in raw_summary.columns:

        raise RuntimeError(
            "raw_encounter_summary exists, but "
            "'duplicate_rows' is missing."
        )


    rows = []


    # V1/V3/V4 are unaffected Encounter controls.
    # V0 was recovered from unaffected domains and does not
    # contain retained Encounter identity in normalized data.
    for variant in [
        "V0",
        "V1",
        "V2",
        "V3",
        "V4",
        "V5",
    ]:

        if variant in set(
            raw_summary["variant"]
            .astype(str)
        ):

            value = int(
                raw_summary.loc[
                    raw_summary[
                        "variant"
                    ].astype(str)
                    ==
                    variant,
                    "duplicate_rows"
                ].iloc[0]
            )


        elif variant == "V0":

            # Use unaffected raw variants as baseline evidence.
            unaffected = (
                raw_summary[
                    raw_summary[
                        "variant"
                    ].isin(
                        [
                            "V1",
                            "V3",
                            "V4",
                        ]
                    )
                ][
                    "duplicate_rows"
                ]
            )

            value = (
                int(
                    unaffected.max()
                )
                if len(unaffected)
                else 0
            )


        elif variant == "V5":

            # V5 combines V1 Patient + V3 Condition.
            # It does not contain the V2 Encounter perturbation.
            value = 0


        else:

            value = 0


        rows.append({
            "variant":
                variant,

            "encounter_duplicate_rows":
                value,

            "resolution_source":
                "raw_fhir_encounter_id",
        })


    v2_duplicate_summary = pd.DataFrame(
        rows
    )


# ------------------------------------------------------------
# No valid V2 evidence available
# ------------------------------------------------------------

else:

    raise RuntimeError(
        "No V2 duplicate-Encounter evidence is currently "
        "available. Expected either "
        "V2_ENCOUNTER_DUPLICATE_SOURCE_INDEX or "
        "raw_encounter_summary."
    )


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(
    "V2 DUPLICATE ENCOUNTER SUMMARY"
)

display(
    v2_duplicate_summary
)


# ------------------------------------------------------------
# Controlled validation
# ------------------------------------------------------------

summary_lookup = (
    v2_duplicate_summary
    .set_index(
        "variant"
    )[
        "encounter_duplicate_rows"
    ]
    .to_dict()
)


v0_dup = int(
    summary_lookup.get(
        "V0",
        0
    )
)

v1_dup = int(
    summary_lookup.get(
        "V1",
        0
    )
)

v2_dup = int(
    summary_lookup.get(
        "V2",
        0
    )
)

v3_dup = int(
    summary_lookup.get(
        "V3",
        0
    )
)

v4_dup = int(
    summary_lookup.get(
        "V4",
        0
    )
)


baseline_max = max(
    v0_dup,
    v1_dup,
    v3_dup,
    v4_dup
)


print(
    "\nBaseline/control maximum duplicate rows:",
    baseline_max
)

print(
    "V2 duplicate rows:",
    v2_dup
)


if v2_dup <= baseline_max:

    raise RuntimeError(
        "V2 duplicate Encounter signal does not "
        "increase above the unaffected controls."
    )


print(
    "\nPASS: V2 duplicate Encounter identity "
    "signal increases above controls."
)


# ------------------------------------------------------------
# Resolution mode
# ------------------------------------------------------------

if (
    "V2_RESOLUTION_MODE"
    not in globals()
):

    if (
        "raw_encounter_summary"
        in globals()
    ):

        V2_RESOLUTION_MODE = (
            "raw_fhir_encounter_id"
        )

    else:

        V2_RESOLUTION_MODE = (
            "normalized_dataframe"
        )


print(
    "V2 resolution mode:",
    V2_RESOLUTION_MODE
)

V2 DUPLICATE ENCOUNTER SUMMARY


,variant,encounter_duplicate_rows,resolution_source
0,V0,0,raw_fhir_encounter_id
1,V1,0,raw_fhir_encounter_id
2,V2,2393,raw_fhir_encounter_id
3,V3,0,raw_fhir_encounter_id
4,V4,0,raw_fhir_encounter_id
5,V5,0,raw_fhir_encounter_id



Baseline/control maximum duplicate rows: 0
V2 duplicate rows: 2393

PASS: V2 duplicate Encounter identity signal increases above controls.
V2 resolution mode: raw_fhir_encounter_id


# Phase D

## Recompute fidelity metrics

In [15]:
LOSS_STATUSES = {Fate.UNMAPPED.value, Fate.DROPPED.value, Fate.TRACEABILITY_LOSS.value}

def metric_rows_for_audit(audit, variant):
    work = audit.copy()
    work["has_warning"] = work["warning_code"].fillna("").astype(str).str.len() > 0
    work["lineage_valid"] = (
        work["is_mapping_event"]
        & work["source_resource_id"].notna()
        & (work["fidelity_status"] != Fate.TRACEABILITY_LOSS.value)
        & work["target_omop_record_id"].notna()
    )
    work["mapped_target"] = work["is_mapping_event"] & work["target_omop_record_id"].notna()
    work["is_loss"] = work["fidelity_status"].isin(LOSS_STATUSES)
    work["is_ambiguity_or_warning"] = (
        (work["fidelity_status"] == Fate.AMBIGUOUS.value)
        | work["has_warning"]
    )

    rows = []
    mapped = work[work["is_mapping_event"]]

    rows.append({
        "variant": variant,
        "source_resource_type": "ALL",
        "target_omop_table": "ALL",
        "audit_items": len(work),
        "mapped_items": len(mapped),
        "lineage_coverage": mapped["lineage_valid"].mean() if len(mapped) else np.nan,
        "source_mapping_coverage": work["mapped_target"].sum()/len(work) if len(work) else np.nan,
        "transformation_loss_rate": work["is_loss"].mean() if len(work) else np.nan,
        "ambiguity_warning_rate": work["is_ambiguity_or_warning"].mean() if len(work) else np.nan,
    })

    for (rtype, table), sub in work.groupby(
        ["source_resource_type","target_omop_table"], dropna=False
    ):
        sub_mapped = sub[sub["is_mapping_event"]]
        rows.append({
            "variant": variant,
            "source_resource_type": str(rtype),
            "target_omop_table": str(table) if pd.notna(table) else "UNMAPPED",
            "audit_items": len(sub),
            "mapped_items": len(sub_mapped),
            "lineage_coverage": sub_mapped["lineage_valid"].mean() if len(sub_mapped) else np.nan,
            "source_mapping_coverage": sub["mapped_target"].sum()/len(sub) if len(sub) else np.nan,
            "transformation_loss_rate": sub["is_loss"].mean() if len(sub) else np.nan,
            "ambiguity_warning_rate": sub["is_ambiguity_or_warning"].mean() if len(sub) else np.nan,
        })

    return pd.DataFrame(rows)

fidelity_metrics = pd.concat(
    [metric_rows_for_audit(TFL_AUDIT[v], v) for v in ["V0","V1","V2","V3","V4","V5"]],
    ignore_index=True
)

overall_metrics = fidelity_metrics[
    (fidelity_metrics["source_resource_type"]=="ALL")
    & (fidelity_metrics["target_omop_table"]=="ALL")
].copy()

display(overall_metrics)

fidelity_metrics.to_csv(
    FIDELITY_METRIC_OUT / "tfl_primary_metrics_corrected.csv",
    index=False
)

,variant,source_resource_type,target_omop_table,audit_items,mapped_items,lineage_coverage,source_mapping_coverage,transformation_loss_rate,ambiguity_warning_rate
0,V0,ALL,ALL,154333,126071,1.0,0.816876,0.183124,0.407878
9,V1,ALL,ALL,154333,126071,1.0,0.816876,0.183124,0.408921
18,V2,ALL,ALL,154333,126071,1.0,0.816876,0.183124,0.407878
27,V3,ALL,ALL,154333,126071,1.0,0.816876,0.183124,0.407878
36,V4,ALL,ALL,151507,126071,1.0,0.832113,0.167887,0.397863
45,V5,ALL,ALL,154333,126071,1.0,0.816876,0.183124,0.408921


In [16]:
warning_rows = []

for variant, audit in TFL_AUDIT.items():
    for row in audit.itertuples():
        for warning in split_warnings(row.warning_code):
            warning_rows.append({
                "variant": variant,
                "source_resource_type": row.source_resource_type,
                "target_omop_table": row.target_omop_table,
                "warning_code": warning,
            })

warning_long = pd.DataFrame(warning_rows)

warning_counts = (
    warning_long.groupby(["variant","warning_code"], dropna=False)
    .size()
    .rename("warning_count")
    .reset_index()
)

audit_denominators = {v: len(TFL_AUDIT[v]) for v in TFL_AUDIT}
warning_counts["audit_items"] = warning_counts["variant"].map(audit_denominators)
warning_counts["warning_rate"] = warning_counts["warning_count"] / warning_counts["audit_items"]

display(warning_counts)

warning_counts.to_csv(
    FIDELITY_METRIC_OUT / "tfl_warning_counts_rates_corrected.csv",
    index=False
)

,variant,warning_code,warning_count,audit_items,warning_rate
0,V0,W_CONFLICTING_CODING,25000,154333,0.161987
1,V0,W_MEDICATION_ATTRIBUTION,9687,154333,0.062767
2,V0,W_UNMAPPED_RESOURCE,28262,154333,0.183124
3,V1,W_CONFLICTING_CODING,25000,154333,0.161987
4,V1,W_DEMOGRAPHIC_MISSING,161,154333,0.001043
5,V1,W_MEDICATION_ATTRIBUTION,9687,154333,0.062767
6,V1,W_UNMAPPED_RESOURCE,28262,154333,0.183124
7,V2,W_CONFLICTING_CODING,25000,154333,0.161987
8,V2,W_MEDICATION_ATTRIBUTION,9687,154333,0.062767
9,V2,W_UNMAPPED_RESOURCE,28262,154333,0.183124


In [17]:
EXPECTED_WARNINGS = {
    "V1": ["W_DEMOGRAPHIC_MISSING"],
    "V2": ["W_DUPLICATE_SOURCE_ID","W_TRACEABILITY_LOSS"],
    "V3": ["W_CONFLICTING_CODING"],
    "V4": ["W_MEDICATION_ATTRIBUTION"],
    "V5": ["W_DEMOGRAPHIC_MISSING","W_CONFLICTING_CODING"],
}

warning_lookup = {
    (r.variant, r.warning_code): (int(r.warning_count), float(r.warning_rate))
    for r in warning_counts.itertuples()
}

rows = []

for variant, expected in EXPECTED_WARNINGS.items():
    for warning in expected:
        v0_count, v0_rate = warning_lookup.get(("V0", warning), (0,0.0))
        variant_count, variant_rate = warning_lookup.get((variant, warning), (0,0.0))
        rows.append({
            "variant": variant,
            "expected_warning": warning,
            "v0_count": v0_count,
            "variant_count": variant_count,
            "delta_count": variant_count-v0_count,
            "v0_rate": v0_rate,
            "variant_rate": variant_rate,
            "delta_rate": variant_rate-v0_rate,
            "increased_vs_v0": variant_rate > v0_rate,
        })

variant_validation = pd.DataFrame(rows)
display(variant_validation)

variant_validation.to_csv(
    FIDELITY_METRIC_OUT / "tfl_variant_warning_validation_corrected.csv",
    index=False
)

variant_gate = (
    variant_validation.groupby("variant")["increased_vs_v0"]
    .all()
    .reset_index(name="all_expected_warnings_increased")
)

display(variant_gate)

,variant,expected_warning,v0_count,variant_count,delta_count,v0_rate,variant_rate,delta_rate,increased_vs_v0
0,V1,W_DEMOGRAPHIC_MISSING,0,161,161,0.000000,0.001043,0.001043,True
1,V2,W_DUPLICATE_SOURCE_ID,0,0,0,0.000000,0.000000,0.000000,False
2,V2,W_TRACEABILITY_LOSS,0,0,0,0.000000,0.000000,0.000000,False
3,V3,W_CONFLICTING_CODING,25000,25000,0,0.161987,0.161987,0.000000,False
4,V4,W_MEDICATION_ATTRIBUTION,9687,9843,156,0.062767,0.064967,0.002200,True
5,V5,W_DEMOGRAPHIC_MISSING,0,161,161,0.000000,0.001043,0.001043,True
6,V5,W_CONFLICTING_CODING,25000,25000,0,0.161987,0.161987,0.000000,False


,variant,all_expected_warnings_increased
0,V1,True
1,V2,False
2,V3,False
3,V4,True
4,V5,False


# Phase E

## Save corrected audit outputs

In [18]:
PATCH_AUDIT_DIR = RUN_ROOT / "fidelity_audit_warningfix_v9"
PATCH_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

for variant, audit in TFL_AUDIT.items():
    audit.to_parquet(
        PATCH_AUDIT_DIR / f"{variant}_fidelity_audit_corrected.parquet",
        index=False
    )

v2_identifier_diagnostics.to_csv(
    FIDELITY_METRIC_OUT / "v2_encounter_identifier_diagnostics.csv",
    index=False
)

v2_duplicate_summary.to_csv(
    FIDELITY_METRIC_OUT / "v2_duplicate_encounter_summary.csv",
    index=False
)

condition_conflict_summary.to_csv(
    FIDELITY_METRIC_OUT / "v3_condition_conflict_summary.csv",
    index=False
)

print("Corrected audits:", PATCH_AUDIT_DIR)
print("Corrected metrics:", FIDELITY_METRIC_OUT)

Corrected audits: /content/drive/MyDrive/fhir_omop_colab/tfl_execution_v6/fidelity_audit_warningfix_v9
Corrected metrics: /content/ohdsi-fhir-omop-showcase-demo/outputs/fidelity_metrics
